<a href="https://colab.research.google.com/github/hilalrusydin95/Hilal/blob/main/hilal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. install dan import library

In [2]:
!pip install -q ipywidgets statsmodels

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import ipywidgets as widgets
from IPython.display import display, clear_output

%matplotlib inline

#2. memuat dataset

In [3]:
url = "https://raw.githubusercontent.com/Alam-A99/DATAAKSI/refs/heads/main/dataset_fiskal_97_hipoFull.csv"
df = pd.read_csv(url)

print("Data berhasil dimuat:", df.shape)
df.head()

Data berhasil dimuat: (1067, 24)


,Kota,Tahun,DBH_PAJAK,DBH_SDA,DBH_Lainnya,DAU_Grant,DAU_Earmark,DAK_Fisik,DAK_Nonfisik,PAD,...,bm_lingkungan,bm_ekonomi,bm_infrastruktur,bm_pariwisata,bm_perlindungan_sosial,KEMISKINAN,PENGANGGURAN,KETIMPANGAN,IPM,PE
0,Kota Banda Aceh,2014,77.91,49.42,26.00,96.98,47.59,83.94,72.02,153.66,...,175.78,253.43,286.18,100.88,133.54,5.31,9.79,0.47,65.31,2.82
1,Kota Banda Aceh,2015,54.88,37.29,35.35,112.77,59.88,74.74,52.84,119.84,...,98.47,285.13,133.72,268.34,254.48,10.50,6.19,0.46,64.99,7.33
2,Kota Banda Aceh,2016,78.91,27.08,39.83,107.56,52.78,96.58,97.41,179.24,...,264.70,212.19,192.71,64.94,59.59,9.57,3.68,0.44,71.00,4.82
3,Kota Banda Aceh,2017,72.77,26.01,46.46,109.93,80.91,63.84,73.98,150.55,...,219.24,288.64,277.40,272.15,267.70,7.77,9.79,0.46,83.49,7.27
4,Kota Banda Aceh,2018,93.06,82.81,31.70,127.89,60.87,77.19,80.88,148.09,...,259.48,87.86,51.55,132.27,178.19,17.43,5.50,0.36,73.57,5.52


#3. widget pilihan (kota, variabel, tombol)

In [4]:
kota_list = sorted(df["Kota"].unique())
var_list = df.select_dtypes(include=[np.number]).drop(columns=["Tahun"]).columns.tolist()

lay = widgets.Layout(width="450px")
sty = {"description_width": "110px"}

dd_kota = widgets.Dropdown(options=kota_list, value="Kota Makassar",
                           description="Kota:", layout=lay, style=sty)
dd_y = widgets.Dropdown(options=var_list, value="DBH_PAJAK",
                        description="Variabel Y:", layout=lay, style=sty)
dd_x1 = widgets.Dropdown(options=var_list, value="bm_pendidikan",
                         description="Variabel X1:", layout=lay, style=sty)
dd_x2 = widgets.Dropdown(options=var_list, value="KETIMPANGAN",
                         description="Variabel X2:", layout=lay, style=sty)
btn = widgets.Button(description="Jalankan Analisis", button_style="success",
                     layout=lay)
out = widgets.Output()

#4. fungsi regresi dan penjelasan tabel hasil

In [5]:
def hitung_regresi(d, y_name, x1_name, x2_name):
    X = sm.add_constant(d[[x1_name, x2_name]])
    model = sm.OLS(d[y_name], X).fit()

    print("TAHAP 1. HASIL REGRESI (OLS)")
    print(model.summary())

    print("\nCara membaca tabel di atas:")
    print("- R-squared    : seberapa besar variasi Y yang bisa dijelaskan model")
    print("- F-statistic  : uji pengaruh X1 dan X2 secara bersama-sama (simultan)")
    print("- coef         : koefisien regresi tiap variabel")
    print("- P>|t|        : p-value uji parsial, dibandingkan dengan alpha 0.05")

    if model.f_pvalue < 0.05:
        print(f"\nUji simultan: Prob (F-statistic) = {model.f_pvalue:.4f} < 0.05, "
              f"jadi {x1_name} dan {x2_name} secara bersama-sama berpengaruh signifikan terhadap {y_name}.")
    else:
        print(f"\nUji simultan: Prob (F-statistic) = {model.f_pvalue:.4f} > 0.05, "
              f"jadi {x1_name} dan {x2_name} secara bersama-sama tidak berpengaruh signifikan terhadap {y_name}.")
    print()
    return model

#5. fungsi interpretasi koefisien, uji t, dan R²

In [6]:
def interpretasi(model, y_name, x1_name, x2_name):
    b0 = model.params["const"]
    b1 = model.params[x1_name]
    b2 = model.params[x2_name]
    p1 = model.pvalues[x1_name]
    p2 = model.pvalues[x2_name]
    r2 = model.rsquared

    print("TAHAP 2. INTERPRETASI")
    print(f"Persamaan regresi:")
    print(f"Y = {b0:.3f} + {b1:.3f} {x1_name} + {b2:.3f} {x2_name}\n")

    print("a. Koefisien")
    print(f"- Konstanta {b0:.3f}: nilai {y_name} ketika {x1_name} dan {x2_name} sama dengan 0")
    print(f"- {x1_name} ({b1:.3f}): kenaikan 1 satuan {x1_name} mengubah {y_name} sebesar {b1:.3f}, dengan {x2_name} dianggap tetap")
    print(f"- {x2_name} ({b2:.3f}): kenaikan 1 satuan {x2_name} mengubah {y_name} sebesar {b2:.3f}, dengan {x1_name} dianggap tetap\n")

    print("b. Uji parsial (alpha 0.05)")
    for nama, p in [(x1_name, p1), (x2_name, p2)]:
        if p < 0.05:
            print(f"- {nama}: p-value {p:.4f} < 0.05, berpengaruh signifikan terhadap {y_name}")
        else:
            print(f"- {nama}: p-value {p:.4f} > 0.05, tidak berpengaruh signifikan terhadap {y_name}")

    print("\nc. Koefisien determinasi")
    print(f"- R-squared = {r2:.4f}, artinya {r2*100:.2f}% variasi {y_name} dijelaskan oleh "
          f"{x1_name} dan {x2_name}, sedangkan {(1-r2)*100:.2f}% dijelaskan faktor lain di luar model.\n")
    return b0, b1, b2, r2

#6. fungsi grafik 3D dan penjelasannya

In [7]:
def grafik_3d(d, kota, y_name, x1_name, x2_name, b0, b1, b2, r2):
    g1 = np.linspace(d[x1_name].min(), d[x1_name].max(), 30)
    g2 = np.linspace(d[x2_name].min(), d[x2_name].max(), 30)
    G1, G2 = np.meshgrid(g1, g2)
    Z = b0 + b1 * G1 + b2 * G2

    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(G1, G2, Z, alpha=0.5, cmap="viridis")
    ax.scatter(d[x1_name], d[x2_name], d[y_name], color="blue", s=40, label="Data aktual")

    for i in range(len(d)):
        ax.text(d[x1_name].iloc[i], d[x2_name].iloc[i], d[y_name].iloc[i],
                str(int(d["Tahun"].iloc[i])), fontsize=8)

    ax.set_xlabel(x1_name)
    ax.set_ylabel(x2_name)
    ax.set_zlabel(y_name)
    ax.set_title(f"Regresi 3D - {kota} (R2 = {r2:.3f})")
    ax.view_init(elev=25, azim=-60)
    ax.legend()

    print("TAHAP 3. GRAFIK 3D")
    plt.show()

    print("\nCara membaca grafik:")
    print("- Titik biru : data aktual tiap tahun (angka di titik = tahun)")
    print("- Bidang     : bidang regresi hasil model")
    print("- Semakin dekat titik ke bidang, semakin baik model menjelaskan data")
    print(f"- Pada data ini R-squared = {r2:.3f}, jadi kecocokan titik terhadap bidang "
          f"{'cukup baik' if r2 >= 0.5 else 'masih lemah'}.")

#7. tombol dan tampilan panel

In [8]:
def analisis(b):
    with out:
        clear_output(wait=True)

        kota = dd_kota.value
        y_name, x1_name, x2_name = dd_y.value, dd_x1.value, dd_x2.value

        if len({y_name, x1_name, x2_name}) < 3:
            print("Variabel Y, X1, dan X2 harus berbeda")
            return

        d = df[df["Kota"] == kota].dropna(subset=[y_name, x1_name, x2_name])
        if len(d) < 5:
            print("Data", kota, "terlalu sedikit untuk regresi:", len(d), "baris")
            return

        model = hitung_regresi(d, y_name, x1_name, x2_name)
        b0, b1, b2, r2 = interpretasi(model, y_name, x1_name, x2_name)
        grafik_3d(d, kota, y_name, x1_name, x2_name, b0, b1, b2, r2)


btn.on_click(analisis)

display(widgets.VBox([dd_kota, dd_y, dd_x1, dd_x2, btn]), out)

Output()